## 1. Objective

Build the operator x municipality activity matrix -- the same activity dimensions as
notebook 09 (event volume, temporal presence, procedural diversity), but grouped by
`(ID_PLANO, CD_MUNICIPIO_BENEFICIARIO, CD_MUNICIPIO_PRESTADOR)` instead of `ID_PLANO`
alone, so operator activity can later be clustered per municipality (notebook 12), not
just per operator overall (notebook 10).

Two municipality columns exist in the treated base and carry different meanings:

- `CD_MUNICIPIO_BENEFICIARIO`: municipality where the **beneficiary/patient resides**
  (demand-side geography).
- `CD_MUNICIPIO_PRESTADOR`: municipality where the **service was provided**
  (supply-side/network geography).

Both are kept as separate grouping keys (not merged into one), because they answer
different business questions and collapsing them would silently discard information.

**Relation to Risk R03**: `documents/extras/target_definition_potencial_pacientes.md`
(Section 5) already warned, independently of this notebook, that adding municipality to
the grouping key "causes almost half the groups to collapse into a single observation --
exactly the 'statistical noise from excessive granularity' risk... documented as Risk
R03". This notebook proceeds with that granularity anyway, per an explicit product
request. The sparsity check below confirms that warning directly with this notebook's
own numbers.

> **Data setup.** This notebook reads local files from `../data/`, which are not versioned (the repository's `.gitignore` blocks data files). No database connection is needed. The inputs are produced by the earlier notebooks from the public ANS open data (see the README, section *Running the Project*). Run `data_pipeline/08_filter_medicina_de_grupo.ipynb` first (or `python scripts/run_pipeline.py` from the repository root). On Google Colab, clone the repository first, install `requirements.txt`, and run the pipeline notebooks so that `data/` is populated in the Colab session.

## 2. Data Loading and Aggregation

The activity matrix is built from the same source as notebook 09 -- the Medicina de Grupo
subset produced by `data_pipeline/08_filter_medicina_de_grupo.ipynb`
(`north_region_mama_2025_data_en_medicina_de_grupo.parquet`) -- but grouped by the operator x
municipality pair instead of by operator alone.

Two of the five original activity metrics from notebook 09 --
`municipios_beneficiarios` and `municipios_prestadores` (distinct-municipality counts)
-- are **not carried over**: at this granularity every group is, by construction, a
single beneficiary municipality and a single provider municipality, so those counts
would be constant (=1) and carry no information. The remaining metrics (event volume,
temporal presence, procedural diversity) still vary meaningfully within an
operator-municipality pair and are kept.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_PATH = Path("../data/processed/north_region_mama_2025_data_en_medicina_de_grupo.parquet")
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        "Missing Medicina de Grupo base. Run data_pipeline/08_filter_medicina_de_grupo.ipynb "
        f"first. Expected path: {INPUT_PATH}"
    )

df_norte = pd.read_parquet(INPUT_PATH)

print("Dataset shape:", df_norte.shape)
print("Unique operators:", df_norte["ID_PLANO"].nunique())
print("Unique beneficiary municipalities:", df_norte["CD_MUNICIPIO_BENEFICIARIO"].nunique())
print("Unique provider municipalities:", df_norte["CD_MUNICIPIO_PRESTADOR"].nunique())

Dataset shape: (70004, 119)
Unique operators: 1052
Unique beneficiary municipalities: 581
Unique provider municipalities: 49


In [2]:
GROUP_KEYS = ["ID_PLANO", "CD_MUNICIPIO_BENEFICIARIO", "CD_MUNICIPIO_PRESTADOR"]

activity_matrix_muni = (
    df_norte
    .groupby(GROUP_KEYS, observed=True)
    .agg(
        total_eventos=("ID_EVENTO_ATENCAO_SAUDE", "nunique"),
        total_itens=("ID_EVENTO_ATENCAO_SAUDE", "size"),
        meses_ativos=("MES", "nunique"),
        procedimentos_distintos=("CD_PROCEDIMENTO", "nunique"),
    )
    .reset_index()
)

print("Rows (operator x municipality pairs):", len(activity_matrix_muni))
activity_matrix_muni.describe()

Rows (operator x municipality pairs): 5089


,total_eventos,total_itens,meses_ativos,procedimentos_distintos
count,5089.000000,5089.000000,5089.000000,5089.000000
mean,13.390057,13.755944,2.978188,1.564158
std,125.396367,126.129872,3.033376,0.981482
min,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000
50%,2.000000,2.000000,2.000000,1.000000
75%,4.000000,5.000000,4.000000,2.000000
max,5747.000000,5747.000000,12.000000,11.000000


### Sparsity check

Unlike the operator-level matrix in notebook 09, this granularity is expected to be
sparser -- most operators are not present in every municipality, so many
(operator, municipality) pairs will have very little activity. This is checked
explicitly before any modeling decision, following the same due diligence notebook 09
applied to its own granularity.

In [3]:
single_event_pairs = (activity_matrix_muni["total_eventos"] == 1).sum()
single_row_pairs = (activity_matrix_muni["total_itens"] == 1).sum()

print(f"Pairs with exactly 1 distinct event: {single_event_pairs} "
      f"({single_event_pairs / len(activity_matrix_muni) * 100:.1f}%)")
print(f"Pairs with exactly 1 item row: {single_row_pairs} "
      f"({single_row_pairs / len(activity_matrix_muni) * 100:.1f}%)")
print()
print(activity_matrix_muni[["total_eventos", "meses_ativos", "procedimentos_distintos"]].describe())

Pairs with exactly 1 distinct event: 2140 (42.1%)
Pairs with exactly 1 item row: 1992 (39.1%)

       total_eventos  meses_ativos  procedimentos_distintos
count    5089.000000   5089.000000              5089.000000
mean       13.390057      2.978188                 1.564158
std       125.396367      3.033376                 0.981482
min         1.000000      1.000000                 1.000000
25%         1.000000      1.000000                 1.000000
50%         2.000000      2.000000                 1.000000
75%         4.000000      4.000000                 2.000000
max      5747.000000     12.000000                11.000000


**Finding**: a large share of operator-municipality pairs are single-event,
single-month occurrences -- an operator showing up once in a given municipality, not a
sustained presence there. This is expected at this granularity (it mirrors the same
long-tail activity pattern already documented in notebook 09, now sliced finer) and is
not treated as noise to remove: a pair with one event is still a real, if minimal, data
point. It does mean the "active" cluster found below should be read as *operator activity
concentrated in that municipality*, not as *the operator's overall size*.

### Manual Aggregation Validation

Following the same due diligence as notebook 09: a sample of (operator, municipality)
pairs is selected and their activity metrics are recalculated directly from the
item-level treated base, then compared against the values in `activity_matrix_muni`, to
verify the group-by aggregation itself was applied correctly before any feature
engineering is built on top of it.

In [4]:
# A blind random sample would, with high probability, land entirely on the
# dominant (1 event, 1 month, 1 procedure) mass (39.1% of all rows -- see the sparsity
# check above), which cannot exercise the nunique()-over-months aggregation logic at
# all. The sample is stratified instead: the single highest-volume pair, one pair with
# multiple active months, one mid-tier pair, and two pairs drawn at random from the rest,
# so the check actually has a chance to catch a bug in any of the four metrics.

top_pair_key = (
    activity_matrix_muni.sort_values("total_eventos", ascending=False)
    [GROUP_KEYS].iloc[[0]]
)
multi_month_key = (
    activity_matrix_muni[activity_matrix_muni["meses_ativos"] > 1]
    .sort_values("meses_ativos", ascending=False)
    [GROUP_KEYS].iloc[[0]]
)
median_events = activity_matrix_muni["total_eventos"].median()
mid_tier_key = (
    activity_matrix_muni.iloc[
        [(activity_matrix_muni["total_eventos"] - median_events).abs().idxmin()]
    ][GROUP_KEYS]
)
random_keys = activity_matrix_muni[GROUP_KEYS].sample(2, random_state=42)

sample_pairs = pd.concat(
    [top_pair_key, multi_month_key, mid_tier_key, random_keys], ignore_index=True
).drop_duplicates()

sample_pairs

,ID_PLANO,CD_MUNICIPIO_BENEFICIARIO,CD_MUNICIPIO_PRESTADOR
0,1173218,130260,130260
1,1169857,150140,150140
2,1169447,150420,150553
3,163444,110020,110012
4,266963,355030,130406


In [5]:
manual_validation = (
    df_norte.merge(sample_pairs, on=GROUP_KEYS, how="inner")
    .groupby(GROUP_KEYS, observed=True)
    .agg(
        total_eventos=("ID_EVENTO_ATENCAO_SAUDE", "nunique"),
        total_itens=("ID_EVENTO_ATENCAO_SAUDE", "size"),
        meses_ativos=("MES", "nunique"),
        procedimentos_distintos=("CD_PROCEDIMENTO", "nunique"),
    )
    .reset_index()
)

manual_validation

,ID_PLANO,CD_MUNICIPIO_BENEFICIARIO,CD_MUNICIPIO_PRESTADOR,total_eventos,total_itens,meses_ativos,procedimentos_distintos
0,1169447,150420,150553,2,2,2,2
1,1169857,150140,150140,121,138,12,4
2,1173218,130260,130260,5747,5747,12,10
3,163444,110020,110012,1,1,1,1
4,266963,355030,130406,8,8,6,2


In [6]:
matrix_validation = activity_matrix_muni.merge(sample_pairs, on=GROUP_KEYS, how="inner")[
    GROUP_KEYS + ["total_eventos", "total_itens", "meses_ativos", "procedimentos_distintos"]
]

comparison = manual_validation.merge(
    matrix_validation,
    on=GROUP_KEYS,
    suffixes=("_manual", "_matrix"),
)

comparison

,ID_PLANO,CD_MUNICIPIO_BENEFICIARIO,CD_MUNICIPIO_PRESTADOR,total_eventos_manual,total_itens_manual,meses_ativos_manual,procedimentos_distintos_manual,total_eventos_matrix,total_itens_matrix,meses_ativos_matrix,procedimentos_distintos_matrix
0,1169447,150420,150553,2,2,2,2,2,2,2,2
1,1169857,150140,150140,121,138,12,4,121,138,12,4
2,1173218,130260,130260,5747,5747,12,10,5747,5747,12,10
3,163444,110020,110012,1,1,1,1,1,1,1,1
4,266963,355030,130406,8,8,6,2,8,8,6,2


In [7]:
for metric in ["total_eventos", "total_itens", "meses_ativos", "procedimentos_distintos"]:
    assert (comparison[f"{metric}_manual"] == comparison[f"{metric}_matrix"]).all(), (
        f"Mismatch found in {metric} -- aggregation logic needs review."
    )

print("Manual validation passed for all sampled operator-municipality pairs.")

Manual validation passed for all sampled operator-municipality pairs.


The aggregation was validated using a stratified check, not a blind random sample: the
top-volume pair, the pair with the most active months, a pair near the median, and two
pairs sampled at random were each independently recalculated from the item-level treated
base and compared against `activity_matrix_muni`. Because the sample deliberately
includes pairs with `meses_ativos > 1`, the check genuinely exercises the
`nunique()`-over-months aggregation, unlike a blind random sample of 5 -- with 39.1% of
all pairs sitting at the degenerate (1, 1, 1) point, a naive random sample has a real
chance of never testing that logic at all. All sampled pairs matched across every
evaluated metric.

### Extreme Activity Patterns

The sparsity check above looked at the low end of the distribution (single-event pairs).
This section looks at the high end, following the same practice as notebook 09, before
the matrix is finalized -- both to sanity-check that the most extreme values are
plausible aggregations and not artifacts, and to document how far the long tail actually
reaches.

In [8]:
top_volume = (
    activity_matrix_muni
    .sort_values("total_eventos", ascending=False)
    [GROUP_KEYS + ["total_eventos", "meses_ativos", "procedimentos_distintos"]]
    .head(10)
)

top_volume

,ID_PLANO,CD_MUNICIPIO_BENEFICIARIO,CD_MUNICIPIO_PRESTADOR,total_eventos,meses_ativos,procedimentos_distintos
215,1173218,130260,130260,5747,12,10
3217,239258,130260,130260,4360,12,10
1790,216532,130260,130260,3137,12,8
1523,214111,130260,130260,2039,12,10
3242,239258,150140,150140,1369,12,7
5067,572290,130260,130260,1201,9,7
1615,214725,130260,130260,906,12,8
5012,570616,130260,130260,886,9,7
3682,239430,150553,150553,883,12,4
129,1171682,130260,130260,855,12,7


In [9]:
activity_matrix_muni[
    ["total_eventos", "meses_ativos", "procedimentos_distintos"]
].quantile([0.50, 0.90, 0.95, 0.99, 1.00])

,total_eventos,meses_ativos,procedimentos_distintos
0.50,2.0,2.0,1.0
0.90,14.0,8.0,3.0
0.95,29.0,11.0,3.0
0.99,177.6,12.0,5.0
1.00,5747.0,12.0,11.0


In [10]:
top_pair = top_volume.iloc[0]
top_pair_id = top_pair[GROUP_KEYS]

top_pair_detail = df_norte.merge(
    top_pair_id.to_frame().T, on=GROUP_KEYS, how="inner"
)

print("Top pair by total_eventos:")
print(top_pair_id.to_dict())
print()
print("Distinct MES (calendar months) covered:", top_pair_detail["MES"].nunique())
print("Distinct CD_PROCEDIMENTO values:", top_pair_detail["CD_PROCEDIMENTO"].nunique())
print("Row count for this pair in the treated base:", len(top_pair_detail))

Top pair by total_eventos:
{'ID_PLANO': '1173218', 'CD_MUNICIPIO_BENEFICIARIO': '130260', 'CD_MUNICIPIO_PRESTADOR': '130260'}

Distinct MES (calendar months) covered: 12
Distinct CD_PROCEDIMENTO values: 10
Row count for this pair in the treated base: 5747


### Interpretation

The distribution is, as expected, strongly right-skewed at this granularity too: the
median operator-municipality pair has only 2 events, while the 99th percentile reaches
into the hundreds and the single most extreme pair (checked individually above) reaches
5,747 events spread across all 12 months. Re-querying the treated base directly for that
specific `(ID_PLANO, CD_MUNICIPIO_BENEFICIARIO, CD_MUNICIPIO_PRESTADOR)` triple reproduces
the same row count and month/procedure coverage found in `activity_matrix_muni`, so this
is a real, internally consistent concentration of activity in a single operator-municipality
pair -- not an aggregation artifact. As in notebook 09, this extreme value is kept rather
than removed, since high concentration of activity in a specific location is directly
relevant to distinguishing a genuinely active local presence from a marginal one, which is
the purpose of this notebook.

## 3. Feature Preparation

Same procedure as notebook 09: check skew, drop redundant/collinear metrics,
log-transform the skewed ones, and standardize.

In [11]:
candidate_features = ["total_eventos", "meses_ativos", "procedimentos_distintos"]

correlation_matrix = activity_matrix_muni[
    candidate_features + ["total_itens"]
].corr()
correlation_matrix

,total_eventos,meses_ativos,procedimentos_distintos,total_itens
total_eventos,1.000000,0.245489,0.395753,0.999559
meses_ativos,0.245489,1.000000,0.657619,0.249584
procedimentos_distintos,0.395753,0.657619,1.000000,0.401052
total_itens,0.999559,0.249584,0.401052,1.000000


`total_itens` and `total_eventos` are, as in notebook 09, extremely highly correlated
(both count activity volume at essentially the same level -- one at the item-row level,
one at the distinct-event level). `total_itens` is dropped in favor of `total_eventos`,
which is the less redundant, more interpretable volume measure, consistent with the
choice already made in notebook 09.

In [12]:
activity_features = candidate_features

skew = activity_matrix_muni[activity_features].skew().sort_values(ascending=False)
skew

total_eventos              31.882647
procedimentos_distintos     2.840696
meses_ativos                1.754062
dtype: float64

In [13]:
# notebook 09 (cell 18) hand-picks which features to log-transform rather than applying
# an automatic threshold -- and deliberately leaves `meses_ativos` un-transformed there.
# An earlier version of this notebook used an automatic `abs(skew) > 1` rule instead,
# which silently logged `meses_ativos` too and broke comparability with notebook 09/10's
# methodology without saying so. The choice is matched by hand instead. On the Medicina
# de Grupo population only `total_eventos` has order-of-magnitude skew (31.88);
# `procedimentos_distintos` (2.84) and `meses_ativos` (1.75) sit close together, so no
# threshold the data draws on its own separates them. `procedimentos_distintos` is
# log-transformed and `meses_ativos` is not because that is what notebook 09 does, and
# keeping the two granularities methodologically comparable is the reason this choice is
# hand-made rather than automatic.
log_features = ["total_eventos", "procedimentos_distintos"]
print("Features to log-transform (matched to notebook 09's choice, not an automatic rule):", log_features)

for col in log_features:
    activity_matrix_muni[f"{col}_LOG"] = np.log1p(activity_matrix_muni[col])

activity_matrix_muni[[f"{col}_LOG" for col in log_features]].skew()

Features to log-transform (matched to notebook 08's choice, not an automatic rule): ['total_eventos', 'procedimentos_distintos']


total_eventos_LOG              2.219664
procedimentos_distintos_LOG    1.512475
dtype: float64

In [14]:
activity_features_final = [f"{col}_LOG" if col in log_features else col for col in activity_features]
activity_features_final

['total_eventos_LOG', 'meses_ativos', 'procedimentos_distintos_LOG']

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_muni = scaler.fit_transform(activity_matrix_muni[activity_features_final])

scaled_activity_muni = pd.DataFrame(X_muni, columns=activity_features_final)
scaled_activity_muni.describe()

,total_eventos_LOG,meses_ativos,procedimentos_distintos_LOG
count,5.089000e+03,5.089000e+03,5.089000e+03
mean,-1.452082e-16,-7.818902e-17,4.733228e-16
std,1.000098e+00,1.000098e+00,1.000098e+00
min,-7.189555e-01,-6.522048e-01,-6.662766e-01
25%,-7.189555e-01,-6.522048e-01,-6.662766e-01
50%,-3.103372e-01,-3.225068e-01,-6.662766e-01
75%,2.044611e-01,3.368893e-01,7.025495e-01
max,7.306436e+00,2.974474e+00,5.382597e+00


In [16]:
activity_matrix_muni_scaled = pd.DataFrame(
    X_muni,
    columns=activity_features_final,
    index=activity_matrix_muni.index,
)

for key in GROUP_KEYS:
    activity_matrix_muni_scaled.insert(0, key, activity_matrix_muni[key].values)
# restore intended column order (GROUP_KEYS were inserted in reverse)
activity_matrix_muni_scaled = activity_matrix_muni_scaled[GROUP_KEYS + activity_features_final]

print("Duplicate (operator, municipality) keys:", activity_matrix_muni_scaled.duplicated(subset=GROUP_KEYS).sum())
print("Missing values:")
print(activity_matrix_muni_scaled[activity_features_final].isna().sum())
print("Shape:", activity_matrix_muni_scaled.shape)

Duplicate (operator, municipality) keys: 0
Missing values:
total_eventos_LOG              0
meses_ativos                   0
procedimentos_distintos_LOG    0
dtype: int64
Shape: (5089, 6)


## 4. Persisting the Matrix

Saved under filenames distinct from notebook 09's operator-level matrix, since the unit
of observation here is the (operator, municipality) triple, not the operator alone.
Notebook 12 loads these two files as-is to run K-Means -- no re-scaling or feature
changes happen there.

In [17]:
raw_output_path = "../data/interim/north_region_mama_2025_activity_matrix_by_municipality.parquet"
scaled_output_path = "../data/processed/north_region_mama_2025_activity_matrix_by_municipality_scaled.parquet"

activity_matrix_muni.to_parquet(raw_output_path, index=False)
activity_matrix_muni_scaled.to_parquet(scaled_output_path, index=False)

print(f"Saved: {raw_output_path}")
print(f"Saved: {scaled_output_path}")
print("Raw shape:", activity_matrix_muni.shape)
print("Scaled shape:", activity_matrix_muni_scaled.shape)

Saved: ../data/interim/north_region_mama_2025_activity_matrix_by_municipality.parquet
Saved: ../data/processed/north_region_mama_2025_activity_matrix_by_municipality_scaled.parquet
Raw shape: (5089, 9)
Scaled shape: (5089, 6)


## References

1. Agência Nacional de Saúde Suplementar (ANS). Dados abertos: TISS Ambulatorial [Internet]. Brasília: ANS; 2026 [cited 2026 Sep 11]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/TISS/AMBULATORIAL/
2. Pedregosa F, Varoquaux G, Gramfort A, Michel V, Thirion B, Grisel O, et al. Scikit-learn: machine learning in Python. J Mach Learn Res. 2011;12:2825-30.
3. James G, Witten D, Hastie T, Tibshirani R. An introduction to statistical learning. New York: Springer; 2013.
